[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/bsheese/225/blob/main/08_data_cleaning/08_9_Exercises.ipynb)

# 08.9: Exercises

These exercises cover the full range of tools from module 08: missing data, type conversions, string cleaning, text splitting and extraction, regular expressions, and date handling. Each exercise gives you a specific task and a dataset to work with. Solutions are hidden in collapsible cells below each exercise.

In [1]:
import pandas as pd
import seaborn as sns

# Titanic (bsheese version — has Name column)
titanic_url = "https://raw.githubusercontent.com/bsheese/CSDS125ExampleData/master/data_titanic.csv"
titanic = pd.read_csv(titanic_url)
titanic.columns = ["survived", "pclass", "name", "sex", "age", "sibsp", "parch", "fare"]

# Titanic (seaborn version — has missing values in age and deck)
titanic_s = sns.load_dataset("titanic")[["survived", "pclass", "sex", "age", "fare", "deck"]].copy()

# MPG dataset
mpg = sns.load_dataset("mpg")

# Taxis dataset
taxis = sns.load_dataset("taxis")[["pickup", "dropoff", "passengers", "distance", "fare", "payment"]].copy()

print("Loaded: titanic, titanic_s, mpg, taxis")

Loaded: titanic, titanic_s, mpg, taxis


---
## Part 1: Missing Data

### Exercise 1

Using `titanic_s`, compute a DataFrame that shows, for each column, the number of missing values and the percentage of missing values (rounded to one decimal). Show only the columns that have at least one missing value, sorted from most missing to least.

Expected result: a DataFrame with 3 columns (`deck`, `age`, `embarked`) sorted by percentage missing.

In [2]:
# your code here

In [3]:
#@title Solution
n = len(titanic_s)
missing = titanic_s.isnull().sum()
pct = (missing / n * 100).round(1)
summary = pd.DataFrame({"missing": missing, "pct_missing": pct})
summary[summary["missing"] > 0].sort_values("pct_missing", ascending=False)

,missing,pct_missing
deck,688,77.2
age,177,19.9


### Exercise 2

Using `titanic_s`, apply `dropna(thresh=)` to keep only rows that have at least 5 non-null values out of the 6 available columns. Print the shape before and after.

In [4]:
# your code here

In [5]:
#@title Solution
print("Before:", titanic_s.shape)
result = titanic_s.dropna(thresh=5)
print("After: ", result.shape)

Before: (891, 6)
After:  (733, 6)


### Exercise 3

Create a Series of seven temperature readings where positions 2, 3, and 5 are `None`: `[68.0, None, None, 72.0, 75.0, None, 74.0]`. Apply `ffill()` and `bfill()` and show the results side by side in a DataFrame.

In [6]:
# your code here

In [7]:
#@title Solution
temps = pd.Series([68.0, None, None, 72.0, 75.0, None, 74.0])
pd.DataFrame({
    "original": temps,
    "ffill":    temps.ffill(),
    "bfill":    temps.bfill()
})

,original,ffill,bfill
0,68.0,68.0,68.0
1,NaN,68.0,72.0
2,NaN,68.0,72.0
3,72.0,72.0,72.0
4,75.0,75.0,75.0
5,NaN,75.0,74.0
6,74.0,74.0,74.0


---
## Part 2: Type Conversions

### Exercise 4

You are given a Series of age values that contains two non-numeric entries: `["22", "35", "unknown", "41", "28", "N/A", "19"]`. Use `pd.to_numeric()` to convert the Series to float, turning the non-numeric entries into `NaN`. Then print the dtype and the count of NaN values.

In [8]:
ages_raw = pd.Series(["22", "35", "unknown", "41", "28", "N/A", "19"])
# your code here

In [9]:
#@title Solution
ages_raw = pd.Series(["22", "35", "unknown", "41", "28", "N/A", "19"])
ages_clean = pd.to_numeric(ages_raw, errors="coerce")
print("dtype:", ages_clean.dtype)
print("NaN count:", ages_clean.isnull().sum())
print(ages_clean)

dtype: float64
NaN count: 2
0    22.0
1    35.0
2     NaN
3    41.0
4    28.0
5     NaN
6    19.0
dtype: float64


### Exercise 5

Using `titanic` (the bsheese version), convert the `pclass` column to the `"category"` dtype. Then use `memory_usage(deep=True)` to compare the memory used by the column before and after the conversion.

In [10]:
# your code here

In [11]:
#@title Solution
before = titanic["pclass"].memory_usage(deep=True)
pclass_cat = titanic["pclass"].astype("category")
after = pclass_cat.memory_usage(deep=True)
print(f"Before (int64): {before} bytes")
print(f"After (category): {after} bytes")

Before (int64): 7228 bytes
After (category): 1043 bytes


---
## Part 3: String Cleaning

### Exercise 6

You are given a Series of city names with inconsistent capitalization and whitespace: `[" new york ", "CHICAGO", "Los Angeles", "  boston", "MIAMI  "]`. Using `.str` methods, normalize this Series so that every entry is in title case with no leading or trailing whitespace.

In [12]:
cities = pd.Series([" new york ", "CHICAGO", "Los Angeles", "  boston", "MIAMI  "])
# your code here

In [13]:
#@title Solution
cities = pd.Series([" new york ", "CHICAGO", "Los Angeles", "  boston", "MIAMI  "])
cities.str.strip().str.title()

0       New York
1        Chicago
2    Los Angeles
3         Boston
4          Miami
dtype: str

### Exercise 7

Using `titanic`, filter the DataFrame to show only passengers whose `name` contains the substring `"Miss"`. How many passengers does that filter return? Show the first five names.

In [14]:
# your code here

In [15]:
#@title Solution
miss_df = titanic[titanic["name"].str.contains("Miss", na=False)]
print(f"Count: {len(miss_df)}")
miss_df["name"].head(5)

Count: 182


2                   Miss. Laina Heikkinen
10         Miss. Marguerite Rut Sandstrom
11                Miss. Elizabeth Bonnell
14    Miss. Hulda Amanda Adolfina Vestrom
22                     Miss. Anna McGowan
Name: name, dtype: str

---
## Part 4: Splitting and Extracting

### Exercise 8

Using `titanic`, extract the social title (e.g., `Mr`, `Mrs`, `Miss`, `Master`) from the `name` column using `str.extract()`. Store the result in a new column called `title`. Then show the five most common titles and their counts.

In [16]:
# your code here

In [17]:
#@title Solution
titanic_ex8 = titanic.copy()
titanic_ex8["title"] = titanic_ex8["name"].str.extract(r'(\w+)\.')
titanic_ex8["title"].value_counts().head(5)

title
Mr        513
Miss      182
Mrs       125
Master     40
Dr          7
Name: count, dtype: int64

### Exercise 9

Using `mpg`, create a new column called `make` that contains only the manufacturer name (the first word of the `name` column). Then compute the average `mpg` for each manufacturer, keeping only manufacturers with at least 5 cars in the dataset. Return the results sorted from highest to lowest average mpg.

In [18]:
# your code here

In [19]:
#@title Solution
mpg_ex9 = mpg.copy()
mpg_ex9["make"] = mpg_ex9["name"].str.split().str.get(0)
(
    mpg_ex9.groupby("make")["mpg"]
    .agg(["mean", "count"])
    .rename(columns={"mean": "avg_mpg", "count": "n"})
    .query("n >= 5")
    .sort_values("avg_mpg", ascending=False)
    .round(1)
)

,avg_mpg,n
make,,
vw,39.0,6
honda,33.8,13
renault,32.9,5
datsun,31.1,23
mazda,30.9,10
volkswagen,29.1,15
fiat,28.9,8
toyota,28.4,25
audi,26.7,7


---
## Part 5: Regular Expressions

### Exercise 10

You are given a Series of fare amounts stored as strings with mixed currency symbols: `["$71.28", "£7.25", "22.00", "$512.33", "8.05"]`. Use `str.replace()` with `regex=True` to strip the currency symbols, then convert the result to float using `pd.to_numeric()`.

In [20]:
fares = pd.Series(["$71.28", "£7.25", "22.00", "$512.33", "8.05"])
# your code here

In [21]:
#@title Solution
fares = pd.Series(["$71.28", "£7.25", "22.00", "$512.33", "8.05"])
cleaned = fares.str.replace(r'[^\d.]', '', regex=True)
pd.to_numeric(cleaned)

0     71.28
1      7.25
2     22.00
3    512.33
4      8.05
dtype: float64

### Exercise 11

You are given a Series of phone numbers in multiple formats: `["(555) 867-5309", "555.123.4567", "5551234567", "800-555-9999", "not a phone"]`. Use `str.replace()` with a regex to normalize them all to 10 consecutive digits. Then use `str.contains()` to identify which entries are valid (exactly 10 digits).

In [22]:
phones = pd.Series(["(555) 867-5309", "555.123.4567", "5551234567", "800-555-9999", "not a phone"])
# your code here

In [23]:
#@title Solution
phones = pd.Series(["(555) 867-5309", "555.123.4567", "5551234567", "800-555-9999", "not a phone"])
digits = phones.str.replace(r'[^\d]', '', regex=True)
valid = digits.str.contains(r'^\d{10}$', regex=True)
pd.DataFrame({"original": phones, "digits": digits, "valid": valid})

,original,digits,valid
0,(555) 867-5309,5558675309,True
1,555.123.4567,5551234567,True
2,5551234567,5551234567,True
3,800-555-9999,8005559999,True
4,not a phone,,False


---
## Part 6: Dates and Times

### Exercise 12

You are given a Series of date strings in US format: `["01/15/2023", "03/22/2023", "07/04/2023", "12/31/2023"]`. Parse them to `datetime64` using `pd.to_datetime()` with an explicit `format=` argument. Then create a DataFrame with additional columns for the month number and the day name (e.g., `"Sunday"`).

In [24]:
date_strings = pd.Series(["01/15/2023", "03/22/2023", "07/04/2023", "12/31/2023"])
# your code here

In [25]:
#@title Solution
date_strings = pd.Series(["01/15/2023", "03/22/2023", "07/04/2023", "12/31/2023"])
dates = pd.to_datetime(date_strings, format="%m/%d/%Y")
pd.DataFrame({
    "date": dates,
    "month": dates.dt.month,
    "day_name": dates.dt.day_name()
})

,date,month,day_name
0,2023-01-15,1,Sunday
1,2023-03-22,3,Wednesday
2,2023-07-04,7,Tuesday
3,2023-12-31,12,Sunday


### Exercise 13

Using `taxis`, compute the duration of each ride in minutes. Store the result in a new column `duration_min`. Then find the average duration by `payment` type, rounded to one decimal place.

In [26]:
# your code here

In [27]:
#@title Solution
taxis_ex13 = taxis.copy()
taxis_ex13["duration_min"] = (taxis_ex13["dropoff"] - taxis_ex13["pickup"]).dt.total_seconds() / 60
taxis_ex13.groupby("payment")["duration_min"].mean().round(1).sort_values(ascending=False)

payment
credit card    15.1
cash           12.6
Name: duration_min, dtype: float64

---
## Part 7: Putting It Together

### Exercise 14 (Challenge)

Write a function `clean_titanic(df)` that takes the raw bsheese Titanic DataFrame and returns a cleaned version. The function must apply at least five of the following steps:

1. Rename `"Siblings/Spouses Aboard"` to `"sibsp"` and `"Parents/Children Aboard"` to `"parch"` (this is already done in the setup, so try loading fresh)
2. Convert `pclass` to `"category"` dtype
3. Extract the social title from `name` into a new column `title`
4. Normalize `sex` with `.str.lower().str.strip()`
5. Create a `fare_bucket` column using `pd.cut()` with bins `[0, 10, 50, 100, 600]` and labels `["low", "mid", "high", "premium"]`
6. Drop the `name` column after extracting the title

After writing the function, call it on the raw Titanic data and show the head of the result.

In [28]:
raw_titanic = pd.read_csv(titanic_url)
# raw_titanic has columns: Survived, Pclass, Name, Sex, Age,
#                          Siblings/Spouses Aboard, Parents/Children Aboard, Fare

def clean_titanic(df):
    # your code here
    pass

# clean_titanic(raw_titanic).head()

In [29]:
#@title Solution
raw_titanic = pd.read_csv(titanic_url)

def clean_titanic(df):
    df = df.rename(columns={
        "Survived":                  "survived",
        "Pclass":                    "pclass",
        "Name":                      "name",
        "Sex":                       "sex",
        "Age":                       "age",
        "Siblings/Spouses Aboard":   "sibsp",
        "Parents/Children Aboard":   "parch",
        "Fare":                      "fare",
    })

    # 1. category dtype for pclass
    df["pclass"] = df["pclass"].astype("category")

    # 2. Extract title from name
    df["title"] = df["name"].str.extract(r'(\w+)\.')

    # 3. Drop name after extraction
    df = df.drop(columns=["name"])

    # 4. Normalize sex
    df["sex"] = df["sex"].str.lower().str.strip()

    # 5. Fare bucket
    df["fare_bucket"] = pd.cut(
        df["fare"],
        bins=[0, 10, 50, 100, 600],
        labels=["low", "mid", "high", "premium"]
    )

    return df.reset_index(drop=True)


clean_titanic(raw_titanic).head()

,survived,pclass,sex,age,sibsp,parch,fare,title,fare_bucket
0,0,3,male,22.0,1,0,7.2500,Mr,low
1,1,1,female,38.0,1,0,71.2833,Mrs,high
2,1,3,female,26.0,0,0,7.9250,Miss,low
3,1,1,female,35.0,1,0,53.1000,Mrs,high
4,0,3,male,35.0,0,0,8.0500,Mr,low
